[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/iamjamesbowden/AG952/blob/main/materials/week07/notebooks/AG952_Week07_Capstone_ReadabilityAndSimilarity.ipynb)

## AG952 | Week 7 | Capstone: Readability and Similarity in Annual Report Disclosures

### Testing the Management Obfuscation Hypothesis on real SEC filings -- Apple Inc. versus General Electric

---

**Learning objective:** This notebook walks through the download and analysis of multiple years of 10-K filings from the SEC EDGAR database. You will compute four textual metrics -- Gunning Fog Index, document length, Loughran and McDonald (2011) sentiment, and cosine similarity -- and assess whether the results are consistent with the Management Obfuscation Hypothesis documented in Li (2008).

**Context:** Apple Inc. (CIK: 320193) serves as a consistently high-performing benchmark. General Electric (CIK: 40987) experienced significant operational and financial difficulties between approximately 2017 and 2019, including large write-downs, earnings misses, and a dividend cut. Li (2008) predicts that firms with lower earnings produce harder-to-read and longer disclosures. This notebook tests that prediction using live filing data.

**Estimated time:** 40--45 minutes (including download time).

---

### Metrics used in this analysis

| Metric | What it captures |
|---|---|
| **Gunning Fog Index** | Estimated reading difficulty; higher scores indicate more complex prose |
| **Document length** | Total word count of the MD&A section; longer disclosures may signal obfuscation |
| **L&M negative sentiment** | Share of words classified as adverse by Loughran and McDonald (2011) |
| **Cosine similarity** | Year-on-year textual overlap; high similarity suggests boilerplate language |

---

Run each cell in order using Shift+Enter.

In [ ]:
#@title Step 1 -- Install and import
!pip install requests beautifulsoup4 matplotlib seaborn pandas numpy scikit-learn -q

import re
import time
import warnings
from collections import Counter

import requests
from bs4 import BeautifulSoup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings("ignore")

print("Step 1 complete -- all libraries imported.")
print()
print("Experiment: Apple Inc. vs. General Electric")
print("Testing the Management Obfuscation Hypothesis (Li, 2008)")
print("Metrics: Gunning Fog Index, document length, L&M sentiment, cosine similarity")


In [ ]:
#@title Step 2 -- Load L&M word lists

LM_NEGATIVE = {
    "abandon", "abandoned", "abandonment", "adverse", "adversely", "adversity",
    "allegation", "allegations", "breach", "breached", "burden", "cease",
    "ceased", "claim", "claims", "complaint", "complaints", "conflict",
    "corrupt", "corruption", "crisis", "critical", "damage", "damages",
    "decline", "declined", "declines", "declining", "default", "defaulted",
    "defaults", "deficit", "deficits", "difficult", "difficulties", "difficulty",
    "discontinued", "dispute", "disputed", "doubtful", "error", "errors",
    "fail", "failed", "failure", "failures", "false", "fine", "fines",
    "fraud", "fraudulent", "harm", "harmful", "impair", "impaired",
    "impairment", "impairments", "inadequate", "inability", "insufficient",
    "lawsuit", "lawsuits", "layoff", "layoffs", "litigation", "loss", "losses",
    "negative", "negatively", "neglect", "negligence", "noncompliance",
    "penalty", "penalties", "poor", "poorly", "problem", "problems",
    "restructuring", "shortage", "shortfall", "terminated", "termination",
    "unable", "volatile", "volatility", "vulnerability", "weak", "weakness",
    "weaknesses", "writedown", "writeoff", "bankrupt", "bankruptcy",
    "downgrade", "downgraded", "downturn", "restatement", "restated",
    "investigation", "misconduct", "insolvency", "insolvent",
    "delinquent", "delinquency",
}

LM_POSITIVE = {
    "achieve", "achieved", "achievement", "achievements", "attain", "attained",
    "beneficial", "beneficially", "benefit", "benefited", "best", "better",
    "efficiencies", "efficient", "efficiently", "enhance", "enhanced",
    "excellent", "exceptional", "favorable", "favorably", "gain", "gained",
    "gains", "growth", "improved", "improvement", "improvements", "improving",
    "innovative", "outstanding", "positive", "positively", "profitable",
    "profitably", "progress", "prosper", "record", "strong", "stronger",
    "strongest", "success", "successes", "successful", "successfully",
    "superior", "surpassed", "value", "valuable", "increase", "increased",
    "increases", "increasing", "strength", "strengths", "opportunity",
    "opportunities", "advantage", "advantages", "confidence", "confident",
    "effective", "effectively", "lead", "leading", "generate", "generates",
    "generating", "expand", "expanding", "expansion", "profitable",
}

print(f"L&M negative list: {len(LM_NEGATIVE):,} words")
print(f"L&M positive list: {len(LM_POSITIVE):,} words")
print("\nStep 2 complete -- L&M word lists loaded.")


In [ ]:
#@title Step 3 -- Define analysis functions

HEADERS = {"User-Agent": "AG952 Strathclyde University ag952course@strath.ac.uk"}
EDGAR_BASE   = "https://data.sec.gov"
ARCHIVES_BASE = "https://www.sec.gov/Archives/edgar/data"


def get_10k_filings(cik, n=3):
    """Return metadata for the n most recent 10-K filings for a given CIK.

    Parameters
    ----------
    cik : int
        Central Index Key from SEC EDGAR.
    n : int
        Maximum number of 10-K filings to return.

    Returns
    -------
    list of dict
        Each dict contains: company, cik, date, year, accession, primary_doc.
    """
    cik_padded = str(cik).zfill(10)
    url = f"{EDGAR_BASE}/submissions/CIK{cik_padded}.json"
    try:
        r = requests.get(url, headers=HEADERS, timeout=15)
        r.raise_for_status()
        data = r.json()
        time.sleep(0.2)
        company = data.get("name", "Unknown")
        recent  = data["filings"]["recent"]
        results = []
        for i, form in enumerate(recent["form"]):
            if form == "10-K" and len(results) < n:
                results.append({
                    "company":     company,
                    "cik":         str(cik),
                    "date":        recent["filingDate"][i],
                    "year":        int(recent["filingDate"][i][:4]),
                    "accession":   recent["accessionNumber"][i],
                    "primary_doc": recent["primaryDocument"][i],
                })
        return sorted(results, key=lambda x: x["year"])
    except Exception as e:
        print(f"Error fetching EDGAR metadata for CIK {cik}: {e}")
        return []


def download_10k_text(filing, max_chars=700000):
    """Download and clean the primary document of a 10-K filing.

    Strips HTML tables, scripts, and iXBRL markup. Returns plain text
    truncated to max_chars characters.
    """
    try:
        acc_clean = filing["accession"].replace("-", "")
        cik_str   = str(int(filing["cik"]))
        url       = f"{ARCHIVES_BASE}/{cik_str}/{acc_clean}/{filing['primary_doc']}"
        r = requests.get(url, headers=HEADERS, timeout=30)
        r.raise_for_status()
        time.sleep(0.2)
        soup = BeautifulSoup(r.text, "html.parser")
        for tag in soup(["table", "script", "style", "ix:header", "ix:nonfraction"]):
            tag.decompose()
        text = soup.get_text(separator=" ")
        text = re.sub(r"\s+", " ", text).strip()
        return text[:max_chars]
    except Exception as e:
        print(f"Error downloading filing ({filing.get('company', '?')}, "
              f"{filing.get('year', '?')}): {e}")
        return ""


def extract_mda(text):
    """Attempt to extract the MD&A section (Item 7) from a 10-K filing.

    Falls back to the first 80,000 characters if no Item 7 section is
    found or the extracted text is under 2,000 characters.
    """
    patterns = [
        r'ITEM\s+7[^A].*?(?=ITEM\s+7A|ITEM\s+8)',
        r'Item\s+7[^A].*?(?=Item\s+7A|Item\s+8)',
    ]
    for pat in patterns:
        m = re.search(pat, text, re.DOTALL | re.IGNORECASE)
        if m and len(m.group()) > 2000:
            return m.group()
    return text[:80000]


def syllable_count(word):
    """Estimate syllable count using a vowel-group heuristic.

    Strips non-alpha characters, counts vowel groups [aeiouy]+,
    and applies adjustments for trailing silent e and ed endings.
    """
    word = re.sub(r'[^a-z]', '', word.lower())
    if not word:
        return 0
    count = len(re.findall(r'[aeiouy]+', word))
    if count > 1 and word.endswith('e') and not word.endswith('le'):
        count -= 1
    if count > 1 and word.endswith('ed'):
        count -= 1
    return max(1, count)


def gunning_fog(text):
    """Compute the Gunning Fog Index for a body of text.

    Fog = 0.4 * (average sentence length + percentage of complex words).
    Complex words are alphabetic words of 3+ characters with 3+ syllables.

    Returns
    -------
    tuple: (fog_score, word_count, avg_sentence_length, pct_complex_words)
    """
    sentences = [s.strip() for s in re.split(r'[.!?]+', text) if len(s.strip()) > 15]
    if not sentences:
        return (0.0, 0, 0.0, 0.0)
    words = re.findall(r'[a-zA-Z]{3,}', text)
    if not words:
        return (0.0, 0, 0.0, 0.0)
    avg_sent_len   = len(words) / len(sentences)
    complex_words  = [w for w in words if syllable_count(w) >= 3]
    pct_complex    = len(complex_words) / len(words) * 100
    fog            = 0.4 * (avg_sent_len + pct_complex)
    return (round(fog, 2), len(words), round(avg_sent_len, 2), round(pct_complex, 2))


def lm_sentiment(text):
    """Compute L&M net sentiment for a body of text.

    Net sentiment = (positive - negative) / (positive + negative).

    Returns
    -------
    tuple: (net_score, pct_negative, pct_positive)
    """
    tokens = re.findall(r'\b[a-zA-Z]+\b', text.lower())
    if not tokens:
        return (0.0, 0.0, 0.0)
    n_neg = sum(1 for t in tokens if t in LM_NEGATIVE)
    n_pos = sum(1 for t in tokens if t in LM_POSITIVE)
    total = len(tokens)
    net   = (n_pos - n_neg) / (n_pos + n_neg) if (n_pos + n_neg) > 0 else 0.0
    return (round(net, 4), round(n_neg / total * 100, 4), round(n_pos / total * 100, 4))


def cosine_sim(text_a, text_b):
    """Compute cosine similarity between two texts using TF-IDF vectors.

    Parameters
    ----------
    text_a, text_b : str

    Returns
    -------
    float: cosine similarity rounded to 4 decimal places
    """
    vec = TfidfVectorizer(max_features=5000, min_df=1)
    try:
        tfidf = vec.fit_transform([text_a, text_b])
        return round(float(cosine_similarity(tfidf[0], tfidf[1])[0][0]), 4)
    except Exception:
        return float("nan")


print("Step 3 complete -- analysis functions defined:")
print("  get_10k_filings    -- fetch filing metadata from EDGAR")
print("  download_10k_text  -- download and strip HTML")
print("  extract_mda        -- extract MD&A section (Item 7)")
print("  syllable_count     -- vowel-group heuristic")
print("  gunning_fog        -- Fog Index computation")
print("  lm_sentiment       -- L&M net sentiment score")
print("  cosine_sim         -- TF-IDF cosine similarity")


In [ ]:
#@title Step 4 -- Download filings from EDGAR
# Note: this cell takes approximately 3-5 minutes to run.
# EDGAR rate limit is 10 requests per second; delays are included.

COMPANIES = {
    "Apple Inc.":       320193,
    "General Electric": 40987,
}

filings_meta = {}
for company, cik in COMPANIES.items():
    meta = get_10k_filings(cik, n=3)
    filings_meta[company] = meta
    dates = [f['date'] for f in meta]
    print(f"{company}: found {len(meta)} filings -- {dates}")

print()
filing_texts = {company: {} for company in COMPANIES}

for company, meta in filings_meta.items():
    for filing in meta:
        yr = filing["year"]
        print(f"Downloading {company} {yr}...", end=" ", flush=True)
        try:
            full_text = download_10k_text(filing)
            if not full_text:
                print("FAILED -- empty response")
                continue
            mda_text  = extract_mda(full_text)
            filing_texts[company][yr] = mda_text
            print(f"done ({len(mda_text):,} chars)")
        except Exception as e:
            print(f"FAILED -- {e}")

print("\nStep 4 complete -- all filings downloaded.")


In [ ]:
#@title Step 5 -- Compute metrics

records = []

for company in COMPANIES:
    years = sorted(filing_texts[company].keys())
    for idx, yr in enumerate(years):
        text = filing_texts[company][yr]
        if not text:
            continue

        fog, wc, asl, pct_cpx = gunning_fog(text)
        net_sent, pct_neg, pct_pos = lm_sentiment(text)

        if idx > 0:
            prev_yr   = years[idx - 1]
            prev_text = filing_texts[company].get(prev_yr, "")
            sim = cosine_sim(prev_text, text) if prev_text else np.nan
        else:
            sim = np.nan

        records.append({
            "Company":       company,
            "Year":          yr,
            "Fog Index":     fog,
            "Word Count":    wc,
            "Avg Sent Len":  asl,
            "% Complex Words": pct_cpx,
            "L&M Net Sent":  net_sent,
            "% Negative":    pct_neg,
            "% Positive":    pct_pos,
            "Cosine Sim":    sim,
        })

df = pd.DataFrame(records)

print("Results:")
print("=" * 90)
header = ("{:<22} {:>6} {:>7} {:>9} {:>7} {:>7} {:>8} {:>8} {:>9}".format(
         "Company", "Year", "Fog", "Words", "AvgSL", "%Cpx", "%Neg", "%Pos", "CosSim"))
print(header)
print("-" * 90)
for _, row in df.iterrows():
    sim_str = f"{row['Cosine Sim']:.4f}" if not pd.isna(row["Cosine Sim"]) else "    n/a"
    print(f"{row['Company']:<22} {row['Year']:>6} {row['Fog Index']:>7.2f} "
          f"{row['Word Count']:>9,} {row['Avg Sent Len']:>7.1f} "
          f"{row['% Complex Words']:>7.1f} {row['% Negative']:>8.4f} "
          f"{row['% Positive']:>8.4f} {sim_str:>9}")
print("=" * 90)

print("\nStep 5 complete -- metrics computed.")


In [ ]:
#@title Step 6 -- Results dashboard

APPLE_COL = "#50fa7b"
GE_COL    = "#ff79c6"
FIG_BG    = "#0d0d1a"
AX_BG     = "#1a1a2e"
GRID_COL  = "#2a2a4a"
TXT_COL   = "#e0e0e0"

COMPANY_COLOURS = {"Apple Inc.": APPLE_COL, "General Electric": GE_COL}
COMPANY_MARKERS = {"Apple Inc.": "o",        "General Electric": "s"}

fig = plt.figure(figsize=(18, 14))
fig.patch.set_facecolor(FIG_BG)
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.42, wspace=0.32)
axes_grid = [fig.add_subplot(gs[r, c]) for r in range(2) for c in range(2)]

panel_specs = [
    ("Fog Index",      "Gunning Fog Index (higher = harder to read)",   None),
    ("Word Count",     "MD&A Word Count (higher = longer disclosure)",  "Words (thousands)"),
    ("% Negative",     "L&M Negative Sentiment (higher = more negative language)", None),
    ("Cosine Sim",     "Cosine Similarity, year-on-year\n(higher = less new information relative to prior year)", None),
]

for ax, (metric, title, ylabel) in zip(axes_grid, panel_specs):
    ax.set_facecolor(AX_BG)
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.tick_params(colors=TXT_COL)
    ax.title.set_color(TXT_COL)
    ax.set_title(title, fontsize=9.5, fontweight="bold", color=TXT_COL, pad=8)
    if ylabel:
        ax.set_ylabel(ylabel, fontsize=9, color=TXT_COL)
    ax.yaxis.grid(True, linestyle=":", color=GRID_COL, linewidth=0.6)
    ax.set_axisbelow(True)

    for company in COMPANIES:
        sub = df[df["Company"] == company].copy()
        if metric == "Cosine Sim":
            sub = sub.dropna(subset=["Cosine Sim"])
        if sub.empty:
            continue
        years  = sub["Year"].tolist()
        vals   = sub[metric].tolist()
        if metric == "Word Count":
            vals = [v / 1000 for v in vals]

        colour = COMPANY_COLOURS[company]
        marker = COMPANY_MARKERS[company]
        ax.plot(years, vals, color=colour, marker=marker,
                linewidth=2.2, markersize=8, label=company)
        for x, y in zip(years, vals):
            if not np.isnan(y):
                ax.annotate(f"{y:.1f}", (x, y), textcoords="offset points",
                            xytext=(0, 10), ha="center", fontsize=7.5, color=colour)

    all_years = df["Year"].unique().tolist()
    ax.set_xticks(sorted(all_years))
    ax.tick_params(axis="x", colors=TXT_COL, labelsize=8)
    ax.tick_params(axis="y", colors=TXT_COL, labelsize=8)
    ax.legend(fontsize=8, facecolor=AX_BG, labelcolor=TXT_COL, framealpha=0.7)

# Panel-specific extras
axes_grid[0].axhline(18, color="#ff5555", linestyle=":", linewidth=1.2, alpha=0.8)
axes_grid[0].text(df["Year"].min() + 0.05, 18.3, "Graduate reading level (>=18)",
                  fontsize=7.5, color="#ff5555", alpha=0.9)

axes_grid[3].set_ylim(0, 1.05)
axes_grid[3].axhline(0.5, color="#888888", linestyle=":", linewidth=1.0, alpha=0.7)
axes_grid[3].text(df["Year"].min() + 0.05, 0.52, "Moderate similarity",
                  fontsize=7.5, color="#888888", alpha=0.8)

fig.suptitle(
    "Apple Inc. vs. General Electric: Readability, Sentiment and Similarity across 10-K Filings",
    fontsize=12, fontweight="bold", color=TXT_COL, y=1.01
)
plt.savefig("readability_similarity_dashboard.png", dpi=150, bbox_inches="tight",
            facecolor=FIG_BG)
plt.show()

print("\nStep 6 complete -- dashboard saved.")


In [ ]:
#@title Step 7 -- Year-on-year changes

print("Year-on-year metric changes")
print("=" * 90)
header = ("{:<22} {:<14} {:>8} {:>12} {:>9} {:>10}".format(
         "Company", "Period", "Fog", "Words", "%Neg", "CosSim"))
print(header)
print("-" * 90)

for company in COMPANIES:
    sub   = df[df["Company"] == company].sort_values("Year").reset_index(drop=True)
    years = sub["Year"].tolist()
    for i in range(1, len(sub)):
        prev = sub.iloc[i - 1]
        curr = sub.iloc[i]
        d_fog  = curr["Fog Index"]   - prev["Fog Index"]
        d_wc   = curr["Word Count"]  - prev["Word Count"]
        d_neg  = curr["% Negative"]  - prev["% Negative"]
        if not pd.isna(curr["Cosine Sim"]):
            d_sim_str = f"{curr['Cosine Sim']:+.4f}"
        else:
            d_sim_str = "     n/a"
        period = f"{int(prev['Year'])} to {int(curr['Year'])}"
        wc_str = f"{d_wc:+,.0f}"
        print(f"{company:<22} {period:<14} {d_fog:>+8.2f} {wc_str:>12} "
              f"{d_neg:>+9.4f} {d_sim_str:>10}")

print("=" * 90)
print("\nStep 7 complete -- year-on-year changes computed.")


In [ ]:
#@title Step 8 -- Passage-level spot check
# Purpose: Moving from aggregate metrics to a readable passage grounds the
# numbers in actual prose. This is a useful sanity check before drawing
# conclusions from the Fog scores computed above.

print("Passage-level spot check -- most recent year per company")
print("=" * 70)

for company in COMPANIES:
    sub   = df[df["Company"] == company].sort_values("Year")
    if sub.empty:
        continue
    latest_yr   = int(sub["Year"].max())
    text        = filing_texts[company].get(latest_yr, "")
    if not text:
        print(f"{company} ({latest_yr}): no text available.")
        continue

    words       = text.split()
    total       = len(words)
    start_idx   = min(2000, total // 4)
    window      = " ".join(words[start_idx: start_idx + 300])
    fog_w, _, asl_w, pct_w = gunning_fog(window)

    print(f"\nCompany           : {company}")
    print(f"Year              : {latest_yr}")
    print(f"Window Fog score  : {fog_w:.2f}")
    print(f"Avg sentence len  : {asl_w:.1f} words")
    print(f"% complex words   : {pct_w:.1f}%")
    print(f"\nExcerpt (first 350 chars):")
    print(f'"{window[:350]}..."')

print("\nStep 8 complete -- passage-level check done.")


## Interpreting the Results

### What Li (2008) predicts

Li (2008) analyses over 50,000 firm-year observations and finds two robust relationships:

- Firms reporting lower earnings produce annual reports with higher Fog scores and longer disclosures.
- Firms with clearer, more readable reports have more persistent future earnings.

Li (2008) interprets these findings as consistent with managers using linguistic complexity to obscure poor results from investors -- the Management Obfuscation Hypothesis.

---

### What to look for in your results

| Observation | Consistent with | Citation |
|---|---|---|
| Fog, word count or negative sentiment rose for GE during its difficult years | Management Obfuscation Hypothesis | Li (2008) |
| Cosine similarity stayed high during GE's difficult period | Boilerplate padding rather than new disclosure | Dyer, Lang and Stice-Lawrence (2017) |
| Apple's metrics remained stable or improved across years | Transparency as a signal of genuine performance | Li (2008) |

---

### Caveats

- Bushee, Gow and Taylor (2018) show that Fog varies systematically with industry and business model complexity. A technology firm will naturally have higher Fog scores than a retailer, regardless of its earnings quality.
- Dyer, Lang and Stice-Lawrence (2017) demonstrate that regulatory requirements explain much of the observed growth in document length. An increase in word count may therefore reflect compliance pressure rather than obfuscation.
- This analysis covers only the MD&A section. Section-level analysis would provide a more precise test of where complexity is concentrated.
- The sample here covers three years per firm. Li (2008) uses over 50,000 firm-year observations to establish statistical significance.

---

### Discussion questions

1. Which metric showed the largest change for GE? What does that suggest about the most likely mechanism -- complexity, length, or negative tone?
2. Is cosine similarity higher for GE or Apple across this period? Is that what you would expect if GE were padding filings with boilerplate?
3. This analysis uses only the MD&A. How might Fog scores differ across sections of the same filing? What does Dyer et al. (2017) predict about this?
4. What additional data would be required to run a properly controlled test of the obfuscation hypothesis -- one that could rule out the alternative explanations in the caveats above?

---

### References

Li, F. (2008) 'Annual report readability, current earnings, and earnings persistence', *Journal of Accounting and Economics*, 45(2--3), pp. 221--247.

Loughran, T. and McDonald, B. (2011) 'When is a liability not a liability? Textual analysis, dictionaries, and 10-Ks', *Journal of Finance*, 66(1), pp. 35--65.

Dyer, T., Lang, M. and Stice-Lawrence, L. (2017) 'The evolution of 10-K textual disclosure: Evidence over three decades', *Journal of Accounting and Economics*, 64(2--3), pp. 221--245.

Bloomfield, R. (2002) 'The incomplete revelation hypothesis and financial reporting', *Accounting Horizons*, 16(3), pp. 233--243.

In [ ]:
#@title Step 9 -- Extension: section-level analysis (optional)
# OPTIONAL -- for students who want to explore further.
# Demonstrates section-level Fog analysis as a more precise test of Li (2008).

SECTION_PATTERNS = {
    "Business (Item 1)":    r'(?:ITEM|Item)\s+1[^A0-9].*?(?=(?:ITEM|Item)\s+1A|(?:ITEM|Item)\s+2)',
    "Risk Factors (Item 1A)": r'(?:ITEM|Item)\s+1A.*?(?=(?:ITEM|Item)\s+2)',
    "MD&A (Item 7)":        r'(?:ITEM|Item)\s+7[^A0-9].*?(?=(?:ITEM|Item)\s+7A|(?:ITEM|Item)\s+8)',
}

print("Section-level Fog analysis -- most recent year per company")
print("Note: full section extraction requires the complete 10-K text.")
print("This cell uses the already-extracted MD&A text, so Items 1 and 1A")
print("will likely not be found. The cell will report this clearly.")
print("=" * 70)

for company in COMPANIES:
    sub = df[df["Company"] == company].sort_values("Year")
    if sub.empty:
        continue
    latest_yr = int(sub["Year"].max())
    text      = filing_texts[company].get(latest_yr, "")
    if not text:
        print(f"\n{company} ({latest_yr}): no text available.")
        continue

    print(f"\n{company} -- {latest_yr}")
    found_any = False
    for section_name, pat in SECTION_PATTERNS.items():
        m = re.search(pat, text, re.DOTALL | re.IGNORECASE)
        if m and len(m.group()) > 500:
            excerpt = m.group()
            fog_s, wc_s, _, _ = gunning_fog(excerpt)
            print(f"  {section_name:<28} -- Fog: {fog_s:.2f}, Words: {wc_s:,}")
            found_any = True
        else:
            print(f"  {section_name:<28} -- section not found in available text")
    if not found_any:
        print("  No sections matched. Consider re-running Step 4 with a larger max_chars.")

print()
print("Methodological note (Dyer et al., 2017):")
print("  Fog is typically highest in Risk Factors and Notes to Financial Statements,")
print("  and lowest in the MD&A. A high aggregate Fog score may therefore reflect")
print("  the relative size of mandatory disclosure sections rather than managerial")
print("  intent. Section-level analysis is a more precise test of Li (2008).")
print("\nStep 9 complete -- section-level analysis done.")
